Data Understanding


In [1]:
#importing libraries
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


#consistent plot styling
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [2]:
#Loading the dataset
raw_path = "../data/raw/enhanced_smart_waste_dataset_v1.csv"
dataset = pd.read_csv(raw_path)

dataset.head(3)


,ItemName,Brand,Category,StockQty,ArrivalDate,ExpiryDate,DaysUntilExpiry,DailySaleAvg,SpoilageChance,SuggestedAction,StoreLocation,IsSpoiled,OnPromotion,TemperatureSensitive,StoreID,Zone,TemperatureSupport,DistanceToNearestStore,AvgDailySaleInNearbyStores
0,Bingo Mad Angles,ITC,Snacks,92,20-06-2025,28-06-2025,0.5,12.1,1.0,donate,Chennai,False,No,False,Chennai_0,NaN,NaN,6.1,13.178196
1,Multigrain Cookies,Sunfeast,Bakery,224,18-06-2025,29-06-2025,1.0,9.0,1.0,donate,Kanpur,True,No,False,Kanpur_0,NaN,NaN,5.0,8.981118
2,Nestlé Yogurt 150g,Nestlé,Dairy,201,23-06-2025,26-06-2025,0.5,20.1,1.0,donate,Mumbai,False,Yes,True,Mumbai_0,West,False,10.3,17.169273


In [3]:
print("Shape (Rows, columns):",dataset.shape)

Shape (Rows, columns): (1000, 19)


Data types

In [4]:
dataset.dtypes

ItemName                          str
Brand                             str
Category                          str
StockQty                        int64
ArrivalDate                       str
ExpiryDate                        str
DaysUntilExpiry               float64
DailySaleAvg                  float64
SpoilageChance                float64
SuggestedAction                   str
StoreLocation                     str
IsSpoiled                        bool
OnPromotion                       str
TemperatureSensitive             bool
StoreID                           str
Zone                              str
TemperatureSupport             object
DistanceToNearestStore        float64
AvgDailySaleInNearbyStores    float64
dtype: object


#### **Missingness Check**

In [5]:
missing_count   = dataset.isna().sum()
missing_pct     = (dataset.isna().mean() * 100).round(1)
missing_summary = pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
Zone,656,65.6
TemperatureSupport,656,65.6


TemperatureSupport and Zone column have about 65.6% missing values. We don't know whats the causation of this missingness, whether is it at random, due to system inputation or not

**Duplicates Check**

In [6]:
print("Fully duplicate rows:", dataset.duplicated().sum())

Fully duplicate rows: 0


The are no duplicates on this dataset

## **Target Variable**

In [7]:
dataset["IsSpoiled"].value_counts()
dataset["IsSpoiled"].value_counts(normalize=True).round(3)

IsSpoiled
False    0.562
True     0.438
Name: proportion, dtype: float64

#### **Cardinality Check**

In [8]:
categorical_cols = dataset.select_dtypes(include=["object","bool"]).columns.tolist()
for c in categorical_cols:
    print(f"{c}: {dataset[c].nunique()} unique values")

ItemName: 15 unique values
Brand: 10 unique values
Category: 5 unique values
ArrivalDate: 15 unique values
ExpiryDate: 24 unique values
SuggestedAction: 4 unique values
StoreLocation: 5 unique values
IsSpoiled: 2 unique values
OnPromotion: 2 unique values
TemperatureSensitive: 2 unique values
StoreID: 1000 unique values
Zone: 2 unique values
TemperatureSupport: 2 unique values


C:\Users\kgaod\AppData\Local\Temp\ipykernel_12208\493151107.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = dataset.select_dtypes(include=["object","bool"]).columns.tolist()


#### **Leakage Check**

In [9]:
print(dataset.groupby("IsSpoiled")["SpoilageChance"].describe())
print()
print(pd.crosstab(dataset["IsSpoiled"], dataset["SuggestedAction"]))

           count      mean       std   min   25%   50%  75%  max
IsSpoiled                                                       
False      562.0  0.673541  0.320848  0.08  0.38  0.69  1.0  1.0
True       438.0  1.000000  0.000000  1.00  1.00  1.00  1.0  1.0

SuggestedAction  discount  donate  monitor  none
IsSpoiled                                       
False                  23     243       96   200
True                    0     438        0     0


### **Findings**

* The dataset contains 1000 rows and 19 columns
* `Zone` and `TemperatureSupport` missingness matches ~ 65%
* There are no duplicate rows
* `SpoilageChance` and `SuggestedAction` are confirmed as leakage columns. They are **excluded from modelling** 
* `ItemName`, `Brand`, `StoreID` are high cardinality identifiers. Usable for grouping/EDA, not as raw model features